<h1 align="left">TSViT - Llama Model Experimentation</h1>

<p align="left">
  ITESM
  
  <a href="https://www.linkedin.com/in/juanrtato/">Juan Ricardo Albarracin B.</a>
  <br>
  <a href="">Luis Ángel Oporto Añacato.</a>
  <br>
  <a href="">David Alexis García Espinosa.</a>
  <br>
  <b>Last updated:</b> <i>25/05/2025</i>
  <br><br>
  <a target="_blank">
    <img src="https://github.com/QData/TextAttack/workflows/Github%20PyTest/badge.svg" alt="Testing">
  </a>
  <a href="https://img.shields.io/badge/version-0.1.0-blue.svg?cacheSeconds=2592000">
    <img src="https://img.shields.io/badge/version-0.1.0-blue.svg?cacheSeconds=2592000" alt="Version" height="18">
  </a>
</p><br>

In [1]:
CONFIG_PATH = '../datalake/config_vtt.json'
MODEL_PATH = '../datalake/TSVIT/best.pth'
import sys
import os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..', 'scripts')))

In [2]:
import ast
import json
import torch
from visiontotext import llamavlm, visiontotextmodel
from tsvit import torch_utils, model_architecture
from pastis24 import get_dataloaders

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.

🦥 Unsloth Zoo will now patch everything to make training faster!


In [3]:
print(torch.__version__)
print(torch.version.cuda)
print(torch.cuda.is_available())

2.7.0+cu128
12.8
True


In [4]:
device_ids = [0]
with open(CONFIG_PATH, 'r') as f:
    config = json.load(f)
device = torch_utils.get_device(device_ids, allow_cpu=False)
with open("../datalake/label_names_en.json", "r") as json_file:
    label_names_en = json.load(json_file)
with open("../datalake/colormap.txt", "r") as txt_file:
    colormap = txt_file.readlines()
colormap = [ast.literal_eval(line.strip().rstrip(',')) for line in colormap]

In [5]:
records = []
dataloaders_vtt = get_dataloaders(config)

Loading PASTIS2SEQUENCE dataset...
PASTIS2SEQUENCE dataset loaded!


In [6]:
encoder = model_architecture.get_model(config, device)
encoder.load_state_dict(torch.load(MODEL_PATH))
encoder.eval()
print(encoder)

TSViT(
  (to_patch_embedding): Sequential(
    (0): Rearrange('b t c (h p1) (w p2) -> (b h w) t (p1 p2 c)', p1=2, p2=2)
    (1): Linear(in_features=40, out_features=128, bias=True)
  )
  (to_temporal_embedding_input): Linear(in_features=366, out_features=128, bias=True)
  (temporal_transformer): Transformer(
    (layers): ModuleList(
      (0-3): 4 x ModuleList(
        (0): PreNorm(
          (norm): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
          (fn): Attention(
            (to_qkv): Linear(in_features=128, out_features=384, bias=False)
            (to_out): Sequential(
              (0): Linear(in_features=128, out_features=128, bias=True)
              (1): Dropout(p=0.0, inplace=False)
            )
          )
        )
        (1): PreNorm(
          (norm): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
          (fn): FeedForward(
            (net): Sequential(
              (0): Linear(in_features=128, out_features=512, bias=True)
              (1): GE

In [7]:
tsvit_llama = llamavlm.LlamaVLM(
    encoder, decoder_model="unsloth/Llama-3.2-1B-Instruct", 
    input_dim=128,
    decoder_dim=2048
).to(device)

==((====))==  Unsloth 2025.5.7: Fast Llama patching. Transformers: 4.51.3.
   \\   /|    NVIDIA RTX 500 Ada Generation Laptop GPU. Num GPUs = 1. Max memory: 3.998 GB. Platform: Windows.
O^O/ \_/ \    Torch: 2.7.0+cu128. CUDA: 8.9. CUDA Toolkit: 12.8. Triton: 3.3.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.30. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


In [8]:
optimizer = torch.optim.Adam(tsvit_llama.parameters(), lr=1e-4)
tsvit_llama.train()

LlamaVLM(
  (encoder): TSViT(
    (to_patch_embedding): Sequential(
      (0): Rearrange('b t c (h p1) (w p2) -> (b h w) t (p1 p2 c)', p1=2, p2=2)
      (1): Linear(in_features=40, out_features=128, bias=True)
    )
    (to_temporal_embedding_input): Linear(in_features=366, out_features=128, bias=True)
    (temporal_transformer): Transformer(
      (layers): ModuleList(
        (0-3): 4 x ModuleList(
          (0): PreNorm(
            (norm): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
            (fn): Attention(
              (to_qkv): Linear(in_features=128, out_features=384, bias=False)
              (to_out): Sequential(
                (0): Linear(in_features=128, out_features=128, bias=True)
                (1): Dropout(p=0.0, inplace=False)
              )
            )
          )
          (1): PreNorm(
            (norm): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
            (fn): FeedForward(
              (net): Sequential(
                (0): Linea

## Without training

In [9]:
def generate_caption(model, input_single, prompt="An image of", max_length=30):
    model.eval()
    model = model.to(device)
    input_single = input_single.to(device)

    with torch.no_grad():
        visual_emb = model.encoder.get_encoder_embeddings(input_single)
        print('visual_emb shape: ', visual_emb.shape)

        #visual_cls = visual_emb.mean(dim=1, keepdim=True)
        #visual_cls = visual_emb[:, :, :]
        visual_cls = visual_emb[:, 0, :]
        #print('visual_cls shape: ', visual_emb[:, -1, :].shape)
        visual_proj = model.adapter(visual_cls).unsqueeze(1)
        print('visual_proj shape: ', visual_proj.shape)

        input_ids = model.tokenizer(prompt, return_tensors="pt", padding=True).input_ids.to(device)
        input_ids = input_ids.expand(visual_cls.shape[0], -1)
        # text_emb = model.decoder.transformer.wte(input_ids)
        # text_emb = model.decoder.generate(*input_ids, cache_implementation="static")

        # inputs = model.tokenizer(prompt, return_tensors="pt", add_special_tokens=True, padding=True, truncation=True)
        # with torch.no_grad():
        #     outputs = model.decoder(input_ids=inputs["input_ids"].to(device), attention_mask=inputs["attention_mask"].to(device))
        # text_emb = outputs.last_hidden_state        
 
        # inputs = model.tokenizer(prompt, padding=True, truncation=False, return_tensors='pt').to(device)
        # with torch.no_grad():
        #     outputs = model.decoder(**inputs, output_hidden_states=True)
        # text_emb = outputs.hidden_states[-1]

        text_emb = model.decoder.model.embed_tokens(input_ids)
        
        print('text_emb shape: ', text_emb.shape)

        input_emb = torch.cat([visual_proj, text_emb], dim=1)

        # Construir attention_mask
        visual_attention_mask = torch.ones(visual_proj.size()[:-1], dtype=torch.long).to(device)  # [B, 1]
        text_attention_mask = (input_ids != model.tokenizer.pad_token_id).long()  # [B, T]
        attention_mask = torch.cat([visual_attention_mask, text_attention_mask], dim=1)  # [B, 1 + T]

        # Definir pad_token_id
        pad_token_id = model.tokenizer.pad_token_id
        if pad_token_id is None:
            pad_token_id = model.tokenizer.eos_token_id

        # outputs = model.decoder.generate(
        #     inputs_embeds=input_emb,
        #     attention_mask=attention_mask,
        #     max_length=max_length,
        #     do_sample=True,
        #     top_k=50,
        #     top_p=0.95,
        #     temperature=1.0,
        #     num_return_sequences=1,
        #     pad_token_id=pad_token_id
        # )
        outputs = model.decoder.generate(
            inputs_embeds=input_emb
        )

    return model.tokenizer.decode(outputs[0], skip_special_tokens=True)

In [ ]:
prompt = "The image shows"
sample_dict, texts, img_path = next(iter(dataloaders_vtt['eval']))
image_sequence = sample_dict['inputs'].to(device)[1].unsqueeze(0)
caption = generate_caption(tsvit_llama, image_sequence, prompt, max_length=50)
caption

## Model Training

In [10]:
def training_step(model, batch, optimizer, device):
    model.train()
    sample_dict, texts, img_path = batch
    image_sequence = sample_dict['inputs'].to(device)
    print(f"Image shape:, {image_sequence.shape}")            # [B, C, H, W] o similar               # [B, T]
    tokenizer = model.tokenizer(texts, padding=True, truncation=True, return_tensors="pt").to(device)
    input_ids = tokenizer.input_ids
    print(f"Input ids shape:, {input_ids.shape}")            # [B, T]

    loss, _ = model(image_sequence, input_ids)
    if optimizer is not None:
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    return loss.item()

In [ ]:
def evaluate(model, dataloader, device):
    model.eval()
    total_loss = 0
    with torch.no_grad():
        for sample in dataloader:
            loss = training_step(model, sample, optimizer=None, device=device)  # sin backprop
            total_loss += loss
    return total_loss / len(dataloader)

num_epochs = 1
for epoch in range(num_epochs):
    print(f"\nEpoch {epoch+1}/{num_epochs}")
    tsvit_llama.train()
    for sample in dataloaders_vtt['train']:
        loss = training_step(tsvit_llama, sample, optimizer, device)
        print(f"Train Loss: {loss:.4f}")
    
    val_loss = evaluate(tsvit_llama, dataloaders_vtt['eval'], device)
    print(f"Eval Loss: {val_loss:.4f}")


Epoch 1/1
Image shape:, torch.Size([24, 60, 24, 24, 11])
Input ids shape:, torch.Size([24, 82])
visual_proj: torch.Size([24, 1, 2048]), text_emb: torch.Size([24, 82, 2048])
Train Loss: 4.6866
Image shape:, torch.Size([24, 60, 24, 24, 11])
Input ids shape:, torch.Size([24, 82])
visual_proj: torch.Size([24, 1, 2048]), text_emb: torch.Size([24, 82, 2048])
Train Loss: 2.5263
Image shape:, torch.Size([24, 60, 24, 24, 11])
Input ids shape:, torch.Size([24, 78])
visual_proj: torch.Size([24, 1, 2048]), text_emb: torch.Size([24, 78, 2048])
Train Loss: 2.5003
Image shape:, torch.Size([24, 60, 24, 24, 11])
Input ids shape:, torch.Size([24, 108])
visual_proj: torch.Size([24, 1, 2048]), text_emb: torch.Size([24, 108, 2048])
Train Loss: 1.5859
Image shape:, torch.Size([24, 60, 24, 24, 11])
Input ids shape:, torch.Size([24, 89])
visual_proj: torch.Size([24, 1, 2048]), text_emb: torch.Size([24, 89, 2048])
Train Loss: 2.0064
Image shape:, torch.Size([24, 60, 24, 24, 11])
Input ids shape:, torch.Size([

In [ ]:
def generate_caption(model, input_single, prompt="An image of", max_length=30):
    model.eval()
    model = model.to(device)
    input_single = input_single.to(device)

    with torch.no_grad():
        visual_emb = model.encoder.get_encoder_embeddings(input_single)
        print('visual_emb shape: ', visual_emb.shape)

        #visual_cls = visual_emb.mean(dim=1, keepdim=True)
        #visual_cls = visual_emb[:, :, :]
        visual_cls = visual_emb[:, 0, :]
        #print('visual_cls shape: ', visual_emb[:, -1, :].shape)
        visual_proj = model.projector(visual_cls).unsqueeze(1)
        print('visual_proj shape: ', visual_proj.shape)

        input_ids = model.tokenizer(prompt, return_tensors="pt", padding=True).input_ids.to(device)
        input_ids = input_ids.expand(visual_cls.shape[0], -1)
        # text_emb = model.decoder.transformer.wte(input_ids)
        text_emb = model.decoder.model.embed_tokens(input_ids)
        print('text_emb shape: ', text_emb.shape)

        input_emb = torch.cat([visual_proj, text_emb], dim=1)

        # Construir attention_mask
        visual_attention_mask = torch.ones(visual_proj.size()[:-1], dtype=torch.long).to(device)  # [B, 1]
        text_attention_mask = (input_ids != model.tokenizer.pad_token_id).long()  # [B, T]
        attention_mask = torch.cat([visual_attention_mask, text_attention_mask], dim=1)  # [B, 1 + T]

        # Definir pad_token_id
        pad_token_id = model.tokenizer.pad_token_id
        if pad_token_id is None:
            pad_token_id = model.tokenizer.eos_token_id

        outputs = model.decoder.generate(
            inputs_embeds=input_emb,
            attention_mask=attention_mask,
            max_length=max_length,
            do_sample=True,
            top_k=50,
            top_p=0.95,
            temperature=1.0,
            num_return_sequences=1,
            pad_token_id=pad_token_id
        )

    return model.tokenizer.decode(outputs[0], skip_special_tokens=True)

In [ ]:
prompt = "The image shows"
sample_dict, texts, img_path = next(iter(dataloaders_vtt['eval']))
image_sequence = sample_dict['inputs'].to(device)[1].unsqueeze(0)
caption = generate_caption(tsvit_llama, image_sequence, prompt, max_length=50)
prompt + caption

In [ ]:
torch.save(tsvit_llama, '../datalake/tsvit_llama_full.pth')